# 🐝 Lab 1.2 — Trust Issues: Calibration & Conformal Prediction

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/varoonsahgal/advancedml/blob/main/notebooks/lab02_calibration_conformal.ipynb)

**Gradient Hive: Advanced ML Week 2026 · Day 1 — Classical, Sharpened**

⏱ **Time:** ~1.5 hours · runs on CPU (Colab free tier is plenty)

### 🎯 Learning objectives
- Build **reliability diagrams, ECE and the Brier score** from scratch and use them to diagnose over-confident models.
- Fix miscalibration with **Platt scaling, isotonic regression and temperature scaling**, and know when each is appropriate.
- Implement **split conformal prediction** for regression and verify its finite-sample coverage guarantee empirically.
- Get **adaptive intervals** with conformalized quantile regression (CQR) and compare conditional coverage.
- Build **classification prediction sets** (LAC and APS), check coverage across $\alpha$, and use the **MAPIE** library in production.

### 🧰 Prerequisites
- Lab 1.1 (or familiarity with gradient-boosted trees in scikit-learn).
- Basic PyTorch (`nn.Module`, an optimizer loop) and basic probability (quantiles, the Beta distribution helps but isn't required).

### 🌍 Why this matters in 2026
In 2026 models make decisions that people act on: loan approvals, triage queues, LLM-agent tool calls with a confidence threshold. A model that says "95% sure" and is right 70% of the time is worse than useless, because it is *confidently* wrong. Calibration makes the numbers mean what they say. Conformal prediction goes further: it wraps **any** model (a GBDT, a neural net, an LLM classifier) and returns sets or intervals with a **distribution-free, finite-sample coverage guarantee**. Conformal methods have gone from a niche statistical topic to standard uncertainty-quantification tooling, with production libraries such as MAPIE.

> 🐝 *Buzz the Backprop Bee says:* run every cell top to bottom. Each **🧪 Your Turn** cell is followed by a **✅ Solution** cell, so the notebook always runs end to end. Try the exercise first, then peek.

## 🗺️ Roadmap

| Part | What you'll do | ⏱ min |
|---|---|---|
| ⚙️ Setup | imports, seeds, device | 3 |
| 1 · Reliability & ECE | build reliability diagrams and ECE from scratch | 12 |
| 2 · Platt & isotonic | fix over-confident binary models | 12 |
| 3 · Temperature scaling | watch an MLP become over-confident (animation), fix it with one scalar | 15 |
| 4 · Split conformal | intervals with a finite-sample guarantee; coverage is Beta-distributed | 18 |
| 5 · CQR | adaptive intervals from quantile regression | 10 |
| 6 · Prediction sets | LAC vs APS for classification, coverage across α | 15 |
| 7 · Limits & tooling | conditional coverage, shift, MAPIE | 5 |

## ⚙️ Setup

In [ ]:
# ⚙️ Setup: run me first!
import os, sys, time, math, random, warnings, importlib.util
FAST = os.environ.get("LAB_FAST", "0") == "1"   # smoke-test mode: shrinks everything

def ensure(pkgs):
    """pip-install packages that are missing from the environment."""
    missing = [pip for pip, mod in pkgs if importlib.util.find_spec(mod) is None]
    if missing:
        print("📦 Installing:", missing)
        get_ipython().run_line_magic("pip", "install -q " + " ".join(missing))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as Fnn
from scipy import stats, optimize
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

HIVE = ["#F5B700", "#1F1B16", "#FF6B5B", "#1FA39A", "#3D4DB7", "#9B5DE5"]
plt.rcParams.update({"axes.prop_cycle": plt.cycler(color=HIVE), "figure.facecolor": "#FFFBEA", "axes.facecolor": "#FFFDF5", "axes.spines.top": False, "axes.spines.right": False, "font.size": 11})
print(f"FAST={FAST} | device={DEVICE} | torch {torch.__version__}")

---
## Part 1 — What does "calibrated" mean? Reliability diagrams & ECE

A binary classifier outputs $\hat p(x) \in [0,1]$. It is **calibrated** if

$$\mathbb{P}\big(Y = 1 \mid \hat p(X) = p\big) = p \quad \text{for all } p.$$

Among all the emails your spam filter scores at 0.8, about 80% should actually be spam. Note that **calibration ≠ accuracy**: a model that predicts the base rate for everyone is perfectly calibrated and useless. We want sharp *and* calibrated predictions.

**Reliability diagram.** Bin predictions into $M$ equal-width bins $B_1..B_M$. For each bin, compare the average prediction $\text{conf}(B_m)$ with the observed frequency $\text{freq}(B_m)$. A calibrated model sits on the diagonal.

**Expected Calibration Error** (Naeini et al., 2015; popularized for deep nets by Guo et al., 2017):
$$\text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{n}\,\big|\text{freq}(B_m) - \text{conf}(B_m)\big|.$$

**Brier score** $= \frac1n\sum_i (\hat p_i - y_i)^2$ is a *proper scoring rule*: it rewards calibration and sharpness together. So does log loss.

⚠️ ECE depends on the binning and is biased for small samples. Treat it as a diagnostic, not a holy number.

In [ ]:
Xb, yb = make_classification(n_samples=9000 if FAST else 30000, n_features=20, n_informative=8, n_redundant=4,
                             flip_y=0.08, class_sep=0.8, random_state=SEED)
Xb_tr, Xb_rest, yb_tr, yb_rest = train_test_split(Xb, yb, train_size=0.4, random_state=SEED)
Xb_cal, Xb_te, yb_cal, yb_te = train_test_split(Xb_rest, yb_rest, train_size=1/3, random_state=SEED)
print("train", Xb_tr.shape, "| calibration", Xb_cal.shape, "| test", Xb_te.shape)

bin_models = {
    "Logistic regression": LogisticRegression(max_iter=2000),
    "Gaussian naive Bayes": GaussianNB(),
    "Overfit boosted trees": HistGradientBoostingClassifier(max_iter=100 if FAST else 300, learning_rate=0.3, max_leaf_nodes=63,
                                                            min_samples_leaf=2, early_stopping=False, random_state=SEED),
}
for name, m in bin_models.items():
    m.fit(Xb_tr, yb_tr)
    print(f"{name:<24} test acc {m.score(Xb_te, yb_te):.3f}")

### 🧪 Your Turn 1 — Reliability bins and ECE
Implement `calibration_bins(p, y, n_bins)`. It should return `(conf, freq, count)` for the **non-empty** bins only. Then implement `ece(p, y, n_bins)`.

*Hints:* edges `np.linspace(0, 1, n_bins+1)`; bin ids `np.searchsorted(edges[1:-1], p)` (this matches sklearn's convention); `np.bincount(ids, weights=..., minlength=n_bins)` gives per-bin sums.

In [ ]:
# 🧪 Your Turn 1
def calibration_bins(p, y, n_bins=10):
    edges = np.linspace(0, 1, n_bins + 1)
    ids = None        # TODO: bin index of each prediction
    count = None      # TODO: number of predictions per bin
    sum_p = None      # TODO: sum of predictions per bin
    sum_y = None      # TODO: sum of labels per bin
    nz = None         # TODO: mask of non-empty bins
    return None, None, None   # TODO: (conf, freq, count) for non-empty bins

def ece(p, y, n_bins=10):
    conf, freq, count = calibration_bins(p, y, n_bins)
    return None       # TODO: weighted mean of |freq - conf|

In [ ]:
# ✅ Solution (peek only after trying!)
def calibration_bins(p, y, n_bins=10):
    p, y = np.asarray(p, float), np.asarray(y, float)
    edges = np.linspace(0, 1, n_bins + 1)
    ids = np.searchsorted(edges[1:-1], p)
    count = np.bincount(ids, minlength=n_bins)
    sum_p = np.bincount(ids, weights=p, minlength=n_bins)
    sum_y = np.bincount(ids, weights=y, minlength=n_bins)
    nz = count > 0
    return sum_p[nz] / count[nz], sum_y[nz] / count[nz], count[nz]

def ece(p, y, n_bins=10):
    conf, freq, count = calibration_bins(p, y, n_bins)
    return float(np.sum(count / count.sum() * np.abs(freq - conf)))

In [ ]:
# ✔️ Checkpoint: match sklearn.calibration.calibration_curve
p_nb = bin_models["Gaussian naive Bayes"].predict_proba(Xb_te)[:, 1]
conf, freq, cnt = calibration_bins(p_nb, yb_te, 10)
sk_freq, sk_conf = calibration_curve(yb_te, p_nb, n_bins=10, strategy="uniform")
assert np.allclose(conf, sk_conf) and np.allclose(freq, sk_freq)
assert abs(ece(np.full(1000, 0.3), np.r_[np.ones(300), np.zeros(700)]) - 0.0) < 1e-12   # base-rate predictor: perfectly calibrated
print("✔️ Bins match sklearn; the base-rate predictor has ECE = 0 (calibrated ≠ useful!)")

In [ ]:
def metrics(p, y):
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return {"ECE": ece(p, y, 15), "Brier": brier_score_loss(y, p), "log loss": log_loss(y, p), "AUC": roc_auc_score(y, p)}

def reliability(ax, p, y, title, n_bins=15, color=None):
    conf, freq, count = calibration_bins(p, y, n_bins)
    edges = np.linspace(0, 1, n_bins + 1)
    nz = np.bincount(np.searchsorted(edges[1:-1], p), minlength=n_bins) > 0
    centers = ((edges[:-1] + edges[1:]) / 2)[nz]
    ax2 = ax.twinx(); ax2.hist(p, bins=n_bins, range=(0, 1), color=HIVE[4], alpha=0.12); ax2.set_yticks([])
    ax.set_zorder(ax2.get_zorder() + 1); ax.patch.set_visible(False)      # bars in front of the histogram
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax.bar(centers, freq, width=1 / n_bins * 0.92, color=color or HIVE[0], alpha=0.9, edgecolor=HIVE[1], lw=0.5, label="observed freq")
    ax.bar(centers, conf - freq, bottom=freq, width=1 / n_bins * 0.92, color=HIVE[2], alpha=0.3, edgecolor=HIVE[2], lw=0.8, label="gap to mean confidence")
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_xlabel("predicted probability")
    ax.set_title(f"{title}\nECE={ece(p, y, n_bins):.3f}", fontsize=10)

probs_raw = {name: m.predict_proba(Xb_te)[:, 1] for name, m in bin_models.items()}
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, (name, p) in zip(axes, probs_raw.items()):
    reliability(ax, p, yb_te, name)
axes[0].set_ylabel("observed frequency of y=1"); axes[0].legend(loc="upper left")
plt.tight_layout(); plt.show()
pd.DataFrame({n: metrics(p, yb_te) for n, p in probs_raw.items()}).T.round(4)

**Reading the diagrams.** Each bar is one confidence bin: its height is the observed frequency and the red box is the gap to the bin's mean prediction. Bars *below* the diagonal on the right and *above* it on the left mean **over-confidence**: predictions are pushed too close to 0 and 1. Naive Bayes is famously overconfident because it multiplies evidence from correlated features as if they were independent. The over-fit booster has memorized label noise, so it is confidently wrong. The faint blue histogram in the background shows *where* predictions live. Overconfident models pile up near 0 and 1.

> **🍯 Key takeaway:** Calibration means "80% means 80%". Diagnose it with reliability diagrams and proper scores (Brier, log loss). ECE is a handy but binning-dependent summary.

---
## Part 2 — Post-hoc fixes: Platt scaling & isotonic regression

Both methods learn a **monotone map** $g: \hat p \mapsto \tilde p$ on a held-out **calibration set**. Never fit it on the training set: the training predictions are exactly the ones the model has overfit.

* **Platt scaling** (Platt, 1999): fit a 1-D logistic regression on the score $s = \operatorname{logit}(\hat p)$:
  $$\tilde p = \sigma(a\,s + b), \qquad (a, b) = \arg\min_{a,b} \sum_i \text{NLL}\big(y_i, \sigma(a s_i + b)\big).$$
  It has only 2 parameters, so it's data-efficient, but it assumes a sigmoid-shaped distortion. $a < 1$ means "shrink overconfident scores towards 0".
* **Isotonic regression** (Zadrozny & Elkan, 2002): fit the best **non-decreasing step function** $g$ (pool-adjacent-violators algorithm). It's non-parametric and can fix any monotone distortion, but it needs more data (≳ 1000 calibration points) and can overfit.

Because both maps are monotone, they (essentially) preserve the **ranking**, so AUC barely moves. Only the *probabilities* change.

### 🧪 Your Turn 2 — Platt scaling from scratch
Minimize the negative log-likelihood of $\sigma(a s + b)$ with `scipy.optimize.minimize`.

*Hints:* the numerically stable NLL is `np.logaddexp(0, z) - y*z` with `z = a*s + b` (that's $\log(1+e^z) - yz$). Start from `x0=[1.0, 0.0]`.

In [ ]:
# 🧪 Your Turn 2
logit = lambda p: np.log(np.clip(p, 1e-7, 1 - 1e-7) / (1 - np.clip(p, 1e-7, 1 - 1e-7)))
sigmoid = lambda z: 1 / (1 + np.exp(-z))

def fit_platt(s, y):
    def nll(params):
        a, b = params
        return None   # TODO: mean NLL of sigmoid(a*s+b) vs y
    a, b = None, None # TODO: optimize.minimize(nll, x0=[1.0, 0.0]).x
    return a, b

In [ ]:
# ✅ Solution (peek only after trying!)
def fit_platt(s, y):
    def nll(params):
        a, b = params
        z = a * s + b
        return np.mean(np.logaddexp(0, z) - y * z)
    a, b = optimize.minimize(nll, x0=[1.0, 0.0], method="L-BFGS-B").x
    return a, b

In [ ]:
# ✔️ Checkpoint: Platt == unregularized 1-D logistic regression on the logit
s_cal = logit(bin_models["Gaussian naive Bayes"].predict_proba(Xb_cal)[:, 1])
a, b = fit_platt(s_cal, yb_cal)
lr_ref = LogisticRegression(C=1e10, max_iter=5000).fit(s_cal[:, None], yb_cal)
print(f"ours: a={a:.4f}, b={b:.4f} | sklearn: a={lr_ref.coef_[0, 0]:.4f}, b={lr_ref.intercept_[0]:.4f}")
assert np.allclose([a, b], [lr_ref.coef_[0, 0], lr_ref.intercept_[0]], atol=2e-3)
assert a < 1, "Naive Bayes is overconfident, so Platt should shrink its logits (a < 1)"
print("✔️ Platt scaling works, and a<1 confirms the overconfidence diagnosis.")

> 🔮 **Predict-then-run:** for each model, which of Platt or isotonic will give the lowest test ECE? Will either change the AUC?

In [ ]:
rows, calibrated = [], {}
for name, m in bin_models.items():
    p_cal, p_te = m.predict_proba(Xb_cal)[:, 1], m.predict_proba(Xb_te)[:, 1]
    a, b = fit_platt(logit(p_cal), yb_cal)
    iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p_cal, yb_cal)
    calibrated[name] = {"raw": p_te, "Platt": sigmoid(a * logit(p_te) + b), "isotonic": iso.predict(p_te)}
    for method, p in calibrated[name].items():
        rows.append({"model": name, "method": method, **metrics(p, yb_te)})
cal_df = pd.DataFrame(rows).set_index(["model", "method"])
display(cal_df.style.format("{:.4f}").background_gradient(subset=["ECE"], cmap="YlOrBr"))

fig, axes = plt.subplots(3, 3, figsize=(14, 12))
for i, (name, d) in enumerate(calibrated.items()):
    for j, (method, p) in enumerate(d.items()):
        reliability(axes[i, j], p, yb_te, f"{name} · {method}", color=[HIVE[0], HIVE[3], HIVE[5]][j])
plt.tight_layout(); plt.show()

**Takeaways 🐝**
* Both fixes slash ECE. Brier and log loss improve too, because they reward calibration.
* AUC is (nearly) untouched: calibration re-labels the confidence levels without re-ordering the predictions. Isotonic can create ties, which nudges AUC a hair.
* Logistic regression was already well calibrated, since it directly minimizes log loss. It has little to fix.
* In scikit-learn you'd use `CalibratedClassifierCV(method="sigmoid" | "isotonic")`. To calibrate an already-fitted model, wrap it in `FrozenEstimator` (sklearn ≥ 1.6).

> **🍯 Key takeaway:** Fit calibrators on **held-out** data. Platt (2 parameters) is data-efficient, isotonic is flexible but data-hungry, and both leave the ranking (AUC) essentially unchanged.

---
## Part 3 — Neural nets & temperature scaling

Guo et al. (2017) observed that modern, high-capacity neural nets are often **over-confident**. Once training accuracy saturates, the cross-entropy keeps dropping by inflating the logit norms. Their fix is almost embarrassingly simple: **divide the logits by a single scalar** $T > 0$ learned on the calibration set.

$$\tilde p(y \mid x) = \operatorname{softmax}\!\big(z(x) / T\big), \qquad T^\star = \arg\min_T \sum_{i \in \text{cal}} -\log \operatorname{softmax}\big(z(x_i)/T\big)_{y_i}.$$

* $T > 1$ softens (fixes over-confidence) and $T < 1$ sharpens.
* Dividing by $T$ doesn't change the $\arg\max$, so **accuracy is unchanged**.
* It's Platt scaling with a shared slope and no bias, extended to $K$ classes. One parameter means it's very hard to overfit.

For multi-class we use **top-label calibration**: the reliability diagram of $\max_k \hat p_k$ (confidence) against "was the argmax correct?".

Let's train a deliberately over-parameterized MLP on a 6-class problem with 10% label noise, and let it overfit.

In [ ]:
Xm, ym = make_classification(n_samples=8000 if FAST else 24000, n_features=30, n_informative=15, n_redundant=5, n_classes=6,
                             n_clusters_per_class=2, flip_y=0.10, class_sep=1.3, random_state=SEED)
Xm_tr, Xm_rest, ym_tr, ym_rest = train_test_split(Xm, ym, train_size=0.45, random_state=SEED, stratify=ym)
Xm_calA, Xm_rest, ym_calA, ym_rest = train_test_split(Xm_rest, ym_rest, train_size=0.15 / 0.55, random_state=SEED, stratify=ym_rest)
Xm_calB, Xm_te, ym_calB, ym_te = train_test_split(Xm_rest, ym_rest, train_size=0.15 / 0.40, random_state=SEED, stratify=ym_rest)
sc = StandardScaler().fit(Xm_tr)
T_ = lambda a: torch.tensor(sc.transform(a), dtype=torch.float32, device=DEVICE)
xt_tr, xt_calA, xt_calB, xt_te = map(T_, [Xm_tr, Xm_calA, Xm_calB, Xm_te])
yt_tr = torch.tensor(ym_tr, device=DEVICE)
K = 6
print(f"train {len(ym_tr)} | calA (temperature) {len(ym_calA)} | calB (conformal) {len(ym_calB)} | test {len(ym_te)}")

mlp = nn.Sequential(nn.Linear(30, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, K)).to(DEVICE)
opt = torch.optim.Adam(mlp.parameters(), lr=1e-3)
EPOCHS = 40 if FAST else 120
hist, snaps = [], []      # snaps: (epoch, test softmax) for the animation below
for ep in range(EPOCHS):
    mlp.train()
    for idx in torch.randperm(len(yt_tr), device=DEVICE).split(128):
        opt.zero_grad(); loss = Fnn.cross_entropy(mlp(xt_tr[idx]), yt_tr[idx]); loss.backward(); opt.step()
    if ep % 5 == 4 or ep == EPOCHS - 1:
        mlp.eval()
        with torch.no_grad():
            ztr, zte = mlp(xt_tr), mlp(xt_te)
            yte_t = torch.tensor(ym_te, device=DEVICE)
            hist.append((ep + 1, Fnn.cross_entropy(ztr, yt_tr).item(), Fnn.cross_entropy(zte, yte_t).item(),
                         (ztr.argmax(1) == yt_tr).float().mean().item(), (zte.argmax(1) == yte_t).float().mean().item()))
            snaps.append((ep + 1, torch.softmax(zte, 1).cpu().numpy()))
hist = np.array(hist)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(hist[:, 0], hist[:, 1], label="train"); axes[0].plot(hist[:, 0], hist[:, 2], label="test"); axes[0].set_title("cross-entropy (NLL)")
axes[1].plot(hist[:, 0], hist[:, 3], label="train"); axes[1].plot(hist[:, 0], hist[:, 4], label="test"); axes[1].set_title("accuracy")
for ax in axes: ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout(); plt.show()
print(f"final test acc {hist[-1, 4]:.3f}, test NLL {hist[-1, 2]:.3f} (it was {hist[:, 2].min():.3f} at its best)")

with torch.no_grad():
    mlp.eval()
    z_calA, z_calB, z_te = (mlp(x).cpu().numpy().astype(np.float64) for x in (xt_calA, xt_calB, xt_te))

### 🎬 Watching over-confidence emerge
Each panel is the top-label reliability diagram on the **test set** at a different epoch. Early on the bars hug the diagonal. As the network memorizes noisy labels, its confidence races to 1.0 while accuracy stays put, and the bars fall further below the diagonal. (The interactive animation below the panels is skipped in FAST mode.)

In [ ]:
def draw_topl(ax, P, y, title):
    conf, correct = P.max(1), (P.argmax(1) == y).astype(float)
    c, fq, n = calibration_bins(conf, correct, 15)
    ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax.bar(c, fq, width=1 / 15 * 0.9, color=HIVE[0], edgecolor=HIVE[1], lw=0.5)
    ax.bar(c, c - fq, bottom=fq, width=1 / 15 * 0.9, color=HIVE[2], alpha=0.3)
    ax.axvline(conf.mean(), color=HIVE[4], lw=2, label=f"mean conf {conf.mean():.2f}")
    ax.axhline(correct.mean(), color=HIVE[3], lw=2, ls=":", label=f"accuracy {correct.mean():.2f}")
    ax.set_xlim(1 / K - 0.02, 1.0); ax.set_ylim(0, 1.02); ax.legend(loc="upper left", fontsize=8)
    ax.set_title(f"{title} · ECE={ece(conf, correct, 15):.3f}", fontsize=10)

pick = sorted(set([0, len(snaps) // 4, len(snaps) // 2, len(snaps) - 1]))
fig, axes = plt.subplots(1, len(pick), figsize=(4.2 * len(pick), 3.9), sharey=True)
for ax, k in zip(np.atleast_1d(axes), pick):
    draw_topl(ax, snaps[k][1], ym_te, f"epoch {snaps[k][0]}")
    ax.set_xlabel("confidence")
np.atleast_1d(axes)[0].set_ylabel("accuracy in bin")
plt.suptitle("The MLP grows over-confident while its accuracy plateaus", y=1.02); plt.tight_layout(); plt.show()

if not FAST:
    from matplotlib import animation
    from IPython.display import HTML
    fig, ax = plt.subplots(figsize=(6.5, 4.4))
    def frame(k):
        ax.clear(); draw_topl(ax, snaps[k][1], ym_te, f"epoch {snaps[k][0]}"); ax.set_xlabel("confidence"); ax.set_ylabel("accuracy in bin")
    anim = animation.FuncAnimation(fig, frame, frames=len(snaps), interval=300)
    plt.close(fig)
    display(HTML(anim.to_jshtml()))

Test NLL went **up** while test accuracy stayed flat. That's the signature of over-confidence: the network is no more wrong than before, just more *sure* of itself.

### 🧪 Your Turn 3 — Fit the temperature
Implement `fit_temperature(logits, labels)` → `T`. Optimize $\log T$ (so $T$ stays positive) to minimize NLL on the calibration logits.

*Hints:* `log_t` is already created for you; `opt_t = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)`; the closure zeroes grads, computes `loss = Fnn.cross_entropy(z / log_t.exp(), y)`, calls `loss.backward()` and returns `loss`; finally `opt_t.step(closure)`.

In [ ]:
# 🧪 Your Turn 3
def fit_temperature(logits, labels):
    z = torch.tensor(logits, dtype=torch.float64); y = torch.tensor(labels)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    # TODO: set up LBFGS on [log_t] and a closure that returns the NLL of z / exp(log_t)
    return float(log_t.exp())

In [ ]:
# ✅ Solution (peek only after trying!)
def fit_temperature(logits, labels):
    z = torch.tensor(logits, dtype=torch.float64); y = torch.tensor(labels)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt_t = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200, line_search_fn="strong_wolfe")
    def closure():
        opt_t.zero_grad()
        loss = Fnn.cross_entropy(z / log_t.exp(), y)
        loss.backward()
        return loss
    opt_t.step(closure)
    return float(log_t.exp())

In [ ]:
softmax = lambda z: np.exp(z - z.max(1, keepdims=True)) / np.exp(z - z.max(1, keepdims=True)).sum(1, keepdims=True)
nll = lambda P, y: -np.mean(np.log(np.clip(P[np.arange(len(y)), y], 1e-12, None)))

T_star = fit_temperature(z_calA, ym_calA)
grid = np.linspace(0.5, 8, 751); grid_nll = [nll(softmax(z_calA / t), ym_calA) for t in grid]
print(f"T* = {T_star:.3f} (grid search: {grid[np.argmin(grid_nll)]:.3f})")
# ✔️ Checkpoint
assert T_star > 1, "an overconfident net needs T > 1"
assert abs(T_star - grid[np.argmin(grid_nll)]) < 0.03
P_raw, P_ts = softmax(z_te), softmax(z_te / T_star)
assert (P_raw.argmax(1) == P_ts.argmax(1)).all(), "temperature scaling must not change predictions"
assert nll(P_ts, ym_te) < nll(P_raw, ym_te)
print("✔️ Temperature found; predictions unchanged; test NLL improved.")

def multiclass_metrics(P, y):
    conf, correct = P.max(1), (P.argmax(1) == y).astype(float)
    onehot = np.eye(P.shape[1])[y]
    return {"acc": correct.mean(), "top-label ECE": ece(conf, correct, 15), "NLL": nll(P, y), "Brier": np.mean(((P - onehot) ** 2).sum(1))}

fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
axes[0].plot(grid, grid_nll, color=HIVE[4]); axes[0].axvline(T_star, ls="--", color=HIVE[2]); axes[0].axvline(1, ls=":", color="gray")
axes[0].set_xlabel("temperature T"); axes[0].set_ylabel("calibration NLL"); axes[0].set_title(f"NLL vs T  (T*={T_star:.2f})")
for ax, (lab, P) in zip(axes[1:], [("raw softmax", P_raw), (f"temperature-scaled (T={T_star:.2f})", P_ts)]):
    reliability(ax, P.max(1), (P.argmax(1) == ym_te).astype(float), "MLP top-label · " + lab, color=HIVE[0] if "raw" in lab else HIVE[3])
    ax.set_xlabel("confidence (max prob)"); ax.set_xlim(1 / K - 0.02, 1)
axes[1].set_ylabel("accuracy")
plt.tight_layout(); plt.show()
pd.DataFrame({"raw": multiclass_metrics(P_raw, ym_te), "temp-scaled": multiclass_metrics(P_ts, ym_te)}).T.round(4)

> **🍯 Key takeaway:** Rising test NLL with flat accuracy is the fingerprint of over-confidence. Temperature scaling fixes it with one scalar and never changes a single prediction.

---
## Part 4 — Split conformal prediction for regression

Calibration gives us *average* honesty. **Conformal prediction** gives a **guarantee**. For any model and any data distribution, as long as the calibration and test points are **exchangeable** (e.g., i.i.d.),

$$1 - \alpha \;\le\; \mathbb{P}\big(Y_{\text{test}} \in \hat C(X_{\text{test}})\big) \;\le\; 1 - \alpha + \tfrac{1}{n+1}\ \ \text{(upper bound if scores have no ties)}.$$

**Split conformal recipe** (Vovk et al., 2005; Lei et al., 2018):
1. Fit any regressor $\hat\mu$ on the training set.
2. On $n$ calibration points compute **nonconformity scores** $s_i = |y_i - \hat\mu(x_i)|$.
3. Let $\hat q$ be the $\lceil (n+1)(1-\alpha) \rceil$-th smallest score. The $+1$ is the finite-sample correction: it accounts for the test point itself.
4. Predict $\hat C(x) = [\hat\mu(x) - \hat q,\; \hat\mu(x) + \hat q]$.

**Why it works (one line):** by exchangeability, the test score is equally likely to land in any of the $n+1$ rank positions among the scores. So $\mathbb{P}(s_{\text{test}} \le \hat q) \ge \lceil (n+1)(1-\alpha)\rceil / (n+1) \ge 1-\alpha$. 🍯 No assumptions about the model being correct!

We'll start with 1-D **heteroscedastic** data (the noise grows with $x$), so we can *see* what's happening.

In [ ]:
def make_hetero(n, rng):
    x = rng.uniform(0, 5, n)
    y = np.sin(2 * x) + 0.3 * x + rng.normal(0, 1, n) * (0.1 + 0.25 * x)
    return x[:, None], y

rng = np.random.default_rng(SEED)
Xh_tr, yh_tr = make_hetero(2000, rng)
Xh_cal, yh_cal = make_hetero(1000, rng)
Xh_te, yh_te = make_hetero(5000, rng)
reg = HistGradientBoostingRegressor(max_iter=200, random_state=SEED).fit(Xh_tr, yh_tr)
print("fitted mean model 🐝")

### 🧪 Your Turn 4 — The conformal quantile
Implement `conformal_quantile(scores, alpha)`. It returns the $k$-th smallest score with $k = \lceil (n+1)(1-\alpha) \rceil$, or `np.inf` if $k > n$ (too few calibration points for that $\alpha$).

In [ ]:
# 🧪 Your Turn 4
def conformal_quantile(scores, alpha):
    n = len(scores)
    k = None   # TODO: ceil((n+1)*(1-alpha))
    # TODO: return np.inf if k > n else the k-th smallest score
    return None

In [ ]:
# ✅ Solution (peek only after trying!)
def conformal_quantile(scores, alpha):
    n = len(scores)
    k = int(np.ceil((n + 1) * (1 - alpha)))
    if k > n:
        return np.inf
    return float(np.sort(scores)[k - 1])

In [ ]:
# ✔️ Checkpoint
s_toy = np.arange(1, 20, dtype=float)                      # n = 19
assert conformal_quantile(s_toy, 0.1) == 18.0              # ceil(20*0.9) = 18
assert conformal_quantile(s_toy, 0.01) == np.inf           # ceil(20*0.99) = 20 > 19
scores_cal = np.abs(yh_cal - reg.predict(Xh_cal))
q_test = conformal_quantile(scores_cal, 0.1)
n_c = len(scores_cal)
assert np.mean(scores_cal <= q_test) >= (1 - 0.1) * (n_c + 1) / n_c - 1e-12   # at least ceil((n+1)(1-α)) scores are ≤ q̂
assert np.mean(scores_cal < q_test) < (1 - 0.1) * (n_c + 1) / n_c              # ...and q̂ is the smallest such score
print("✔️ conformal_quantile is correct")

ALPHA = 0.1
qhat = conformal_quantile(scores_cal, ALPHA)
mu_te = reg.predict(Xh_te)
covered = np.abs(yh_te - mu_te) <= qhat
print(f"q̂ = {qhat:.3f} | empirical test coverage = {covered.mean():.3f} (target ≥ {1 - ALPHA})")

> 🔮 **Predict-then-run:** coverage on *one* calibration/test split is a random variable. Rerun the whole thing 1000 times with fresh calibration sets of $n = 1000$. What will the **spread** of coverages look like? (Hint: Vovk (2012) showed that, conditional on the calibration set, coverage follows a **Beta** distribution.)

In [ ]:
# Coverage is random: Beta(n+1-l, l) with l = floor((n+1)*alpha)   (Vovk 2012; Angelopoulos & Bates 2021)
R = 200 if FAST else 1000
n_cal = 1000
pool_x, pool_y = make_hetero(n_cal + 20000, np.random.default_rng(123))
pool_s = np.abs(pool_y - reg.predict(pool_x))
covs = []
for r in range(R):
    perm = np.random.default_rng(r).permutation(len(pool_s))
    cal_s, test_s = pool_s[perm[:n_cal]], pool_s[perm[n_cal:n_cal + 3000]]
    covs.append(np.mean(test_s <= conformal_quantile(cal_s, ALPHA)))
covs = np.array(covs)
l = int(np.floor((n_cal + 1) * ALPHA)); beta = stats.beta(n_cal + 1 - l, l)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.hist(covs, bins=30, density=True, color=HIVE[0], alpha=0.8, label=f"{R} reruns")
xx = np.linspace(covs.min() - 0.01, covs.max() + 0.01, 300)
ax.plot(xx, beta.pdf(xx), color=HIVE[4], lw=2, label=f"Beta({n_cal + 1 - l}, {l}) theory")
ax.axvline(1 - ALPHA, ls="--", color=HIVE[2], label="nominal 1-α")
ax.set_xlabel("empirical test coverage"); ax.legend(); ax.set_title("Conformal coverage across calibration sets")
plt.show()
print(f"mean coverage {covs.mean():.4f} (theory {beta.mean():.4f}); 5th-95th pct: {np.percentile(covs, [5, 95]).round(4)}")
assert abs(covs.mean() - beta.mean()) < 0.01   # ✔️ Checkpoint: the guarantee holds on average

Coverage is **≥ 90% on average**, but any single deployment can land a bit below. The spread shrinks like $1/\sqrt{n}$, so collect more calibration data if you need tight guarantees.

Now the big caveat. The guarantee is **marginal**: it holds averaged over all $x$, not for each $x$. A constant-width band on heteroscedastic data will over-cover where noise is small and under-cover where it's big:

In [ ]:
def coverage_by_bin(x, covered, bins):
    ids = np.digitize(x, bins[1:-1])
    return np.array([covered[ids == b].mean() for b in range(len(bins) - 1)])

order = np.argsort(Xh_te[:, 0]); xs = Xh_te[order, 0]
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [2, 1]})
ax = axes[0]
ax.scatter(Xh_te[:, 0], yh_te, s=3, c=np.where(covered, HIVE[3], HIVE[2]), alpha=0.5)
ax.plot(xs, mu_te[order], color=HIVE[1], lw=2, label="μ̂(x)")
ax.fill_between(xs, mu_te[order] - qhat, mu_te[order] + qhat, color=HIVE[0], alpha=0.3, label="split conformal 90%")
ax.set_title("Split conformal: constant width (red = missed)"); ax.legend(loc="upper left"); ax.set_xlabel("x")
xbins = np.linspace(0, 5, 11)
cov_split = coverage_by_bin(Xh_te[:, 0], covered, xbins)
axes[1].bar(xbins[:-1] + 0.25, cov_split, width=0.45, color=HIVE[0]); axes[1].axhline(1 - ALPHA, ls="--", color=HIVE[2])
axes[1].set_ylim(0.5, 1.02); axes[1].set_xlabel("x bin"); axes[1].set_title("conditional coverage by x")
plt.tight_layout(); plt.show()

### Real data check: California housing
Same recipe on real data, now for several $\alpha$ values. We expect empirical coverage ≈ nominal every time.

In [ ]:
def load_housing():
    try:
        from sklearn.datasets import fetch_california_housing
        d = fetch_california_housing()
        return d.data, d.target, "California housing"
    except Exception as e:
        print("⚠️ download failed:", e, "-> synthetic fallback")
        from sklearn.datasets import make_friedman1
        X, y = make_friedman1(n_samples=20640, n_features=8, noise=1.0, random_state=0)
        return X, y, "Friedman #1 (synthetic)"

Xc, yc, CAL_NAME = load_housing()
if FAST:
    idx = np.random.default_rng(0).choice(len(yc), 5000, replace=False); Xc, yc = Xc[idx], yc[idx]
Xc_tr, Xc_rest, yc_tr, yc_rest = train_test_split(Xc, yc, train_size=0.5, random_state=SEED)
Xc_cal, Xc_te, yc_cal, yc_te = train_test_split(Xc_rest, yc_rest, train_size=0.4, random_state=SEED)
reg_c = HistGradientBoostingRegressor(max_iter=300, random_state=SEED).fit(Xc_tr, yc_tr)
s_c = np.abs(yc_cal - reg_c.predict(Xc_cal)); mu_c = reg_c.predict(Xc_te)
tab = []
for a in [0.05, 0.1, 0.2, 0.3]:
    q = conformal_quantile(s_c, a)
    tab.append({"alpha": a, "nominal": 1 - a, "empirical coverage": np.mean(np.abs(yc_te - mu_c) <= q), "interval width ($100k)": 2 * q})
print(CAL_NAME, "| n_cal =", len(yc_cal))
pd.DataFrame(tab).round(3)

> **🍯 Key takeaway:** Split conformal turns *any* model into intervals with a finite-sample guarantee: take the $\lceil (n+1)(1-\alpha) \rceil$-th smallest calibration score. Coverage holds on average over calibration sets, and any single run varies (Beta-distributed).

---
## Part 5 — Conformalized Quantile Regression (CQR)

Fix the constant-width problem by making the *base model* adaptive, then let conformal fix its *coverage* (Romano, Patterson & Candès, 2019):

1. Train two **quantile regressors** $\hat q_{\alpha/2}(x)$ and $\hat q_{1-\alpha/2}(x)$ with the pinball loss $\rho_\tau(u) = u(\tau - \mathbb{1}[u<0])$.
2. Calibration score: how far outside the band $y$ falls (negative if inside):
   $$E_i = \max\big(\hat q_{\alpha/2}(x_i) - y_i,\; y_i - \hat q_{1-\alpha/2}(x_i)\big).$$
3. $\hat Q = $ conformal quantile of $E$, and the interval is $[\hat q_{\alpha/2}(x) - \hat Q,\; \hat q_{1-\alpha/2}(x) + \hat Q]$.

Quantile regressors alone have **no** coverage guarantee: they are often too narrow out of sample. CQR keeps their *shape* and fixes their *level*. $\hat Q$ can be negative, which shrinks the band if the quantile models were too wide.

### 🧪 Your Turn 5 — Implement CQR
Fill in `cqr_intervals`. The quantile models are already fitted for you.

*Hint:* `np.maximum(lo_cal - y_cal, y_cal - hi_cal)` gives the scores.

In [ ]:
# 🧪 Your Turn 5
# (the two quantile regressors are fitted for you)
q_lo = HistGradientBoostingRegressor(loss="quantile", quantile=ALPHA / 2, max_iter=200, random_state=SEED).fit(Xh_tr, yh_tr)
q_hi = HistGradientBoostingRegressor(loss="quantile", quantile=1 - ALPHA / 2, max_iter=200, random_state=SEED).fit(Xh_tr, yh_tr)

def cqr_intervals(q_lo, q_hi, X_cal, y_cal, X_test, alpha):
    lo_cal, hi_cal = q_lo.predict(X_cal), q_hi.predict(X_cal)
    E = None            # TODO: CQR conformity scores
    Q = None            # TODO: conformal quantile of E
    lo, hi = None, None # TODO: adjusted test interval
    return lo, hi

In [ ]:
# ✅ Solution (peek only after trying!)
def cqr_intervals(q_lo, q_hi, X_cal, y_cal, X_test, alpha):
    lo_cal, hi_cal = q_lo.predict(X_cal), q_hi.predict(X_cal)
    E = np.maximum(lo_cal - y_cal, y_cal - hi_cal)
    Q = conformal_quantile(E, alpha)
    lo, hi = q_lo.predict(X_test) - Q, q_hi.predict(X_test) + Q
    return lo, hi

In [ ]:
lo, hi = cqr_intervals(q_lo, q_hi, Xh_cal, yh_cal, Xh_te, ALPHA)
cov_cqr = (yh_te >= lo) & (yh_te <= hi)
raw_cov = np.mean((yh_te >= q_lo.predict(Xh_te)) & (yh_te <= q_hi.predict(Xh_te)))
print(f"raw quantile band coverage: {raw_cov:.3f} | CQR coverage: {cov_cqr.mean():.3f} | split coverage: {covered.mean():.3f}")
print(f"mean width: CQR {np.mean(hi - lo):.3f} vs split {2 * qhat:.3f}")
# ✔️ Checkpoint
assert cov_cqr.mean() >= 1 - ALPHA - 0.02
print("✔️ CQR keeps the marginal guarantee")

cov_cqr_bins = coverage_by_bin(Xh_te[:, 0], cov_cqr, xbins)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), gridspec_kw={"width_ratios": [2, 1]})
ax = axes[0]
ax.scatter(Xh_te[:, 0], yh_te, s=3, c=np.where(cov_cqr, HIVE[3], HIVE[2]), alpha=0.5)
ax.fill_between(xs, mu_te[order] - qhat, mu_te[order] + qhat, color=HIVE[0], alpha=0.25, label="split conformal")
ax.fill_between(xs, lo[order], hi[order], color=HIVE[4], alpha=0.25, label="CQR")
ax.set_title("CQR adapts its width to the local noise"); ax.legend(loc="upper left"); ax.set_xlabel("x")
w = 0.22
axes[1].bar(xbins[:-1] + 0.25 - w / 2, cov_split, width=w, color=HIVE[0], label="split")
axes[1].bar(xbins[:-1] + 0.25 + w / 2, cov_cqr_bins, width=w, color=HIVE[4], label="CQR")
axes[1].axhline(1 - ALPHA, ls="--", color=HIVE[2]); axes[1].set_ylim(0.5, 1.02); axes[1].legend(loc="lower left")
axes[1].set_title("conditional coverage by x"); axes[1].set_xlabel("x bin")
plt.tight_layout(); plt.show()

> **🍯 Key takeaway:** CQR keeps the *shape* of quantile regression and lets conformal fix the *level*. You get adaptive widths and still keep the guarantee.

---
## Part 6 — Conformal prediction sets for classification

For classification we output a **set** $\hat C(x) \subseteq \{1..K\}$ with $\mathbb{P}(Y \in \hat C(X)) \ge 1-\alpha$. A big set means "the model is unsure here", which is useful for triage and human-in-the-loop routing.

**LAC / "least ambiguous" sets** (Sadinle, Lei & Wasserman, 2019): score $s = 1 - \hat p_y(x)$. Include every class with $\hat p_k(x) \ge 1 - \hat q$. This gives the smallest *average* set size, but coverage can be uneven: easy inputs get over-covered and hard ones under-covered.

**APS, adaptive prediction sets** (Romano, Sesia & Candès, 2020): sort the classes by probability and add up mass until you reach the true label:
$$s(x, y) = \sum_{k:\ \hat p_k(x) > \hat p_y(x)} \hat p_k(x) \;+\; U \cdot \hat p_y(x), \qquad U \sim \text{Uniform}(0,1).$$
The set is $\{k : s(x,k) \le \hat q\}$, using the same $U$ for every class of a given input. Uncertain inputs (flat softmax) get bigger sets automatically. The random $U$ smooths over the jumps in the cumulative sums, as in the original paper. Setting $U = 0$ gives a deterministic variant that is just as valid, because we use the same score for calibration and for building sets. It always contains the top class, so it is never empty. (The common shortcut of calibrating with the *inclusive* mass $U = 1$ but then also adding "the class that crosses $\hat q$" is valid but conservative, because the score and the set no longer match.)

Both are valid **whatever** the probabilities are, even uncalibrated ones! Better-calibrated probabilities just give *smaller, more useful* sets. We use the temperature-scaled MLP and the held-out `calB` split. That split wasn't used to fit $T$, so exchangeability holds exactly.

In [ ]:
P_calB, P_test = softmax(z_calB / T_star), softmax(z_te / T_star)

def lac_scores(P, y):  return 1 - P[np.arange(len(y)), y]
def lac_sets(P, qhat): return P >= 1 - qhat

### 🧪 Your Turn 6 — APS scores and sets
Implement `aps_all_scores(P)`, which returns an $(n, K)$ matrix whose entry $[i, k]$ is the **inclusive** cumulative sorted mass up to and including class $k$: $\sum_{j: \hat p_j \ge \hat p_k} \hat p_j$. The randomized score then just subtracts $u \cdot \hat p_k$. (Since $1-U$ is also uniform, that's the formula above.)

*Hints:* `order = np.argsort(-P, axis=1)`; `cums = np.cumsum(np.take_along_axis(P, order, 1), 1)`; then scatter back with `np.put_along_axis(out, order, cums, axis=1)`.

In [ ]:
# 🧪 Your Turn 6
def aps_all_scores(P):
    out = np.zeros_like(P)
    # TODO: sort classes by descending prob, cumulative-sum, scatter back to original class positions
    return out

In [ ]:
# ✅ Solution (peek only after trying!)
def aps_all_scores(P):
    order = np.argsort(-P, axis=1)
    cums = np.cumsum(np.take_along_axis(P, order, axis=1), axis=1)
    out = np.empty_like(P)
    np.put_along_axis(out, order, cums, axis=1)
    return out

In [ ]:
rng_aps = np.random.default_rng(SEED)
def aps_scores(P, y, randomized=True):
    u = rng_aps.uniform(size=len(y)) if randomized else np.ones(len(y))
    idx = np.arange(len(y))
    return aps_all_scores(P)[idx, y] - u * P[idx, y]
def aps_sets(P, qhat, randomized=True):
    u = rng_aps.uniform(size=(len(P), 1)) if randomized else np.ones((len(P), 1))
    return aps_all_scores(P) - u * P <= qhat

# ✔️ Checkpoint: compare with a slow, loop-based reference on a few rows, then check coverage
def aps_score_slow(p, y):
    return sum(pk for pk in p if pk >= p[y])
for i in range(25):
    for k in range(K):
        assert abs(aps_all_scores(P_test[i:i + 1])[0, k] - aps_score_slow(P_test[i], k)) < 1e-9
q_aps = conformal_quantile(aps_scores(P_calB, ym_calB), 0.1)
cov_aps = aps_sets(P_test, q_aps)[np.arange(len(ym_te)), ym_te].mean()
q_det = conformal_quantile(aps_scores(P_calB, ym_calB, randomized=False), 0.1)
S_det = aps_sets(P_test, q_det, randomized=False)
assert abs(cov_aps - 0.9) < 0.03
print(f"✔️ APS matches the reference. Randomized APS coverage at α=0.1: {cov_aps:.3f}")
print(f"   (deterministic version, U=0 in the formula: coverage {S_det[np.arange(len(ym_te)), ym_te].mean():.3f}, avg size {S_det.sum(1).mean():.2f}, never empty)")

> 🔮 **Predict-then-run:** which method has smaller sets on average? Which has more *even* coverage between confident and unconfident inputs?

In [ ]:
alphas = np.array([0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4])
res = {"LAC": [], "APS": []}
for a in alphas:
    for name, (sfn, setfn) in {"LAC": (lac_scores, lac_sets), "APS": (aps_scores, aps_sets)}.items():
        q = conformal_quantile(sfn(P_calB, ym_calB), a)
        S = setfn(P_test, q)
        res[name].append((S[np.arange(len(ym_te)), ym_te].mean(), S.sum(1).mean()))
res = {k: np.array(v) for k, v in res.items()}

fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
axes[0].plot(1 - alphas, 1 - alphas, "--", color="gray", label="nominal")
for name, c in [("LAC", HIVE[0]), ("APS", HIVE[4])]:
    axes[0].plot(1 - alphas, res[name][:, 0], "o-", color=c, label=name)
    axes[1].plot(1 - alphas, res[name][:, 1], "o-", color=c, label=name)
axes[0].set_xlabel("target coverage 1-α"); axes[0].set_ylabel("empirical coverage"); axes[0].legend(); axes[0].set_title("Coverage tracks the target")
axes[1].set_xlabel("target coverage 1-α"); axes[1].set_ylabel("average set size"); axes[1].legend(); axes[1].set_title("Price of certainty: set size")

# conditional coverage by model confidence (difficulty)
a = 0.1
S_lac = lac_sets(P_test, conformal_quantile(lac_scores(P_calB, ym_calB), a))
S_aps = aps_sets(P_test, conformal_quantile(aps_scores(P_calB, ym_calB), a))
conf_bins = np.quantile(P_test.max(1), np.linspace(0, 1, 6))
for j, (name, S, c) in enumerate([("LAC", S_lac, HIVE[0]), ("APS", S_aps, HIVE[4])]):
    hit = S[np.arange(len(ym_te)), ym_te]
    cb = coverage_by_bin(P_test.max(1), hit, conf_bins)
    axes[2].bar(np.arange(5) + (j - 0.5) * 0.38, cb, width=0.38, color=c, label=name)
axes[2].axhline(1 - a, ls="--", color=HIVE[2]); axes[2].set_ylim(0.5, 1.02)
axes[2].set_xticks(range(5), ["hardest\n20%", "", "", "", "easiest\n20%"]); axes[2].legend(loc="lower right")
axes[2].set_title("coverage by model confidence (α=0.1)")
for name, S in [("LAC", S_lac), ("APS", S_aps)]:
    cb = coverage_by_bin(P_test.max(1), S[np.arange(len(ym_te)), ym_te], conf_bins)
    print(f"{name}: avg size {S.sum(1).mean():.2f} | coverage across difficulty quintiles {cb.round(3)} (spread {cb.max() - cb.min():.3f})")
plt.tight_layout(); plt.show()

sizes = pd.DataFrame({"LAC": np.bincount(S_lac.sum(1), minlength=K + 1), "APS": np.bincount(S_aps.sum(1), minlength=K + 1)})
sizes.index.name = "set size"
print("Set-size histogram at α=0.1 (number of test points):"); display(sizes.T)

**What you should see 🐝**
* Both methods hit the target coverage at every $\alpha$. That's the guarantee at work.
* **LAC has the smallest sets.** That's no accident: Sadinle et al. (2019) show LAC minimizes expected set size among valid set predictors when the probabilities are correct. Its threshold is the same for every input, so how evenly it covers depends entirely on how well calibrated the model is in each region.
* **APS pays for adaptivity with bigger sets.** It keeps adding classes until it has covered enough *mass*, so flat, uncertain softmaxes always get big sets, while confident inputs get singletons. The long tail of low-probability classes inflating sets is what motivated RAPS (Boss Level 1).
* Check the right-hand plot for *your* run. The coverage-by-confidence bars are a cheap **conditional-coverage audit**. If one difficulty slice sits well below the dashed line, a marginally valid method is still failing those users.
* LAC (and randomized APS) can return **empty sets**. These count as misses, and in practice you'd fall back to top-1. Size-$K$ sets mean "I have no idea", which is useful as an escalation signal.

> **🍯 Key takeaway:** Prediction sets turn uncertainty into an action signal (big set → escalate). LAC minimizes average size; APS adapts to flat softmaxes. Always audit coverage by slice.

---
## Part 7 — Conditional coverage, distribution shift, and production tooling

**The limits.** Exact *conditional* coverage $\mathbb{P}(Y \in \hat C(X) \mid X = x) \ge 1-\alpha$ for every $x$ is **impossible** to guarantee distribution-free with finite samples without trivially wide sets (Vovk, 2012; Barber et al., 2021). What we *can* do:
* **Adaptive scores** (CQR, APS, normalized residuals $|y-\hat\mu|/\hat\sigma(x)$) approximate conditional coverage in practice.
* **Mondrian / group-conditional conformal:** calibrate separately per group (class, region, demographic). You get a guarantee *within each group*, at the price of needing enough calibration data per group.
* **Distribution shift breaks exchangeability.** Use *weighted* conformal under covariate shift (Tibshirani et al., 2019), or *adaptive conformal inference* for online/time-series settings (Gibbs & Candès, 2021), and **monitor coverage in production**.

**Calibration vs conformal, a cheat sheet**

| | Calibration (Platt / isotonic / temperature) | Conformal prediction |
|---|---|---|
| Output | better probabilities | sets / intervals |
| Guarantee | none (asymptotic at best) | finite-sample, marginal, distribution-free |
| Needs | held-out calibration set | held-out calibration set + exchangeability |
| Use when | you consume probabilities (expected-value decisions, ranking with thresholds) | you need "give me the plausible answers at 90%" |

They compose nicely: calibrate first, then conformalize, and you get valid *and* small sets.

### Production option: MAPIE
[MAPIE](https://github.com/scikit-learn-contrib/MAPIE) (scikit-learn-contrib) implements split, cross and jackknife+ conformal, CQR, LAC/APS/RAPS sets, time-series methods and risk control behind a scikit-learn-style API. The cell below (optional, needs `pip install mapie`) checks that MAPIE's split-conformal interval matches ours. The v1 API names used here: `SplitConformalRegressor(..., confidence_level=, prefit=True)` → `.conformalize()` → `.predict_interval()`.

In [ ]:
# Optional: production conformal with MAPIE (v1+ API). Skips gracefully if unavailable.
try:
    ensure([("mapie", "mapie")])
    from mapie.regression import SplitConformalRegressor
    scr = SplitConformalRegressor(estimator=reg_c, confidence_level=0.9, prefit=True)
    scr.conformalize(Xc_cal, yc_cal)
    _, y_int = scr.predict_interval(Xc_te)
    lo_m, hi_m = y_int[:, 0, 0], y_int[:, 1, 0]
    q_ours = conformal_quantile(s_c, 0.1)
    print(f"MAPIE coverage {np.mean((yc_te >= lo_m) & (yc_te <= hi_m)):.3f} | "
          f"mean width MAPIE {np.mean(hi_m - lo_m):.4f} vs ours {2 * q_ours:.4f}")
    print("max |MAPIE bound - ours| =", float(np.max(np.abs(lo_m - (mu_c - q_ours)))))
except Exception as e:
    print("ℹ️ MAPIE demo skipped:", repr(e)[:200])

> **🍯 Key takeaway:** The guarantee is marginal and relies on exchangeability. Under drift, switch to weighted or adaptive conformal and monitor coverage in production.

---
## 🏆 Boss Level

1. **RAPS.** Implement *regularized* APS (Angelopoulos, Bates, Malik & Jordan, 2021), which adds a penalty $\lambda\,(\text{rank}(y) - k_{reg})^+$ to the APS score. Tune $\lambda, k_{reg}$ on a held-out split, then compare average set size and coverage-by-difficulty against LAC and APS.
2. **Mondrian conformal.** Calibrate the LAC quantile **separately per class**. Plot per-class coverage before and after, and look at what happens to the rare classes' set sizes.
3. **Conformal under shift.** Train on $x \sim U(0, 4)$ and test on $x \sim U(1, 5)$ in the heteroscedastic example. Watch coverage break, then repair it with **weighted conformal** using known density-ratio weights $w(x) = p_{test}(x)/p_{train}(x)$ (Tibshirani et al., 2019).

## 🍯 Key Takeaways
* **Calibrated = honest probabilities.** Check reliability diagrams + Brier/log loss; treat ECE as a diagnostic.
* **Calibrate on held-out data**: Platt (sigmoid), isotonic (monotone step), temperature (one scalar for nets). Accuracy and ranking are preserved.
* **Over-confidence fingerprint:** test NLL up, accuracy flat.
* **Split conformal**: one quantile of calibration scores gives finite-sample, distribution-free *marginal* coverage for any model.
* **Coverage is random** across calibration sets (Beta); more calibration data → tighter.
* **Adaptive scores** (CQR, APS) improve conditional behaviour; **Mondrian** gives per-group guarantees.
* **Exchangeability is the whole assumption.** Under shift, use weighted/adaptive conformal and monitor.

## 🎤 Explain it to a friend
Explain to a friend the difference between a weather app that says "70% chance of rain" and actually rains on 70% of those days, and one that gives you a temperature *range* it promises will contain tomorrow's high 9 days out of 10. Which one is "calibration" and which is "conformal prediction"? Why does the second one only need a pile of past forecasts and their errors, not any knowledge of how the forecast model works?

## 💼 Interview corner
**Q1. A classifier has 95% accuracy but an ECE of 0.15. Is that a problem, and how would you fix it?**
<details><summary>Answer</summary>

It's a problem whenever the probabilities are *used*: thresholds, expected-cost decisions, ranking across models, or human trust. Fit temperature scaling (nets) or Platt/isotonic on a held-out calibration split and re-check reliability and proper scores. Accuracy won't change, because the argmax is preserved.
</details>

**Q2. What exactly does the split-conformal guarantee promise, and what doesn't it promise?**
<details><summary>Answer</summary>

$\mathbb{P}(Y \in \hat C(X)) \ge 1-\alpha$, averaged over the random calibration set *and* test point, assuming exchangeability. It does **not** promise coverage for each $x$ or each subgroup (conditional coverage). It doesn't promise coverage under distribution shift, and it doesn't promise that one particular calibration set achieves $1-\alpha$: that coverage follows a Beta distribution around the target.
</details>

**Q3. Why not fit the temperature on the training set?**
<details><summary>Answer</summary>

The network has (over)fit the training set, so training logits are far more confident *and* more often correct than on new data. The NLL-optimal temperature there would be ≈1 or even <1, which would leave the over-confidence in place or make it worse. Calibration must see data the model hasn't memorized.
</details>

## 🐝 Fun fact
Weather forecasters pioneered probability calibration. The Brier score was introduced in 1950 by meteorologist Glenn W. Brier to verify precipitation forecasts. In the 1970s, Allan Murphy and Robert Winkler found that U.S. National Weather Service probability-of-precipitation forecasts were remarkably well calibrated, a standard many ML models still don't meet.

## 📚 Further reading
* C. Guo, G. Pleiss, Y. Sun, K. Q. Weinberger (2017). *On Calibration of Modern Neural Networks.* ICML.
* J. Platt (1999). *Probabilistic Outputs for Support Vector Machines and Comparisons to Regularized Likelihood Methods.* Advances in Large Margin Classifiers.
* B. Zadrozny & C. Elkan (2002). *Transforming Classifier Scores into Accurate Multiclass Probability Estimates.* KDD.
* A. Niculescu-Mizil & R. Caruana (2005). *Predicting Good Probabilities with Supervised Learning.* ICML.
* G. W. Brier (1950). *Verification of Forecasts Expressed in Terms of Probability.* Monthly Weather Review 78.
* M. P. Naeini, G. Cooper, M. Hauskrecht (2015). *Obtaining Well Calibrated Probabilities Using Bayesian Binning.* AAAI.
* V. Vovk, A. Gammerman, G. Shafer (2005). *Algorithmic Learning in a Random World.* Springer.
* J. Lei, M. G'Sell, A. Rinaldo, R. J. Tibshirani, L. Wasserman (2018). *Distribution-Free Predictive Inference for Regression.* JASA.
* Y. Romano, E. Patterson, E. Candès (2019). *Conformalized Quantile Regression.* NeurIPS.
* Y. Romano, M. Sesia, E. Candès (2020). *Classification with Valid and Adaptive Coverage.* NeurIPS.
* M. Sadinle, J. Lei, L. Wasserman (2019). *Least Ambiguous Set-Valued Classifiers with Bounded Error Levels.* JASA.
* V. Vovk (2012). *Conditional Validity of Inductive Conformal Predictors.* ACML.
* A. N. Angelopoulos & S. Bates (2021). *A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification.* arXiv:2107.07511.
* R. F. Barber, E. Candès, A. Ramdas, R. J. Tibshirani (2021). *The limits of distribution-free conditional predictive inference.* Information and Inference.
* R. J. Tibshirani, R. F. Barber, E. Candès, A. Ramdas (2019). *Conformal Prediction Under Covariate Shift.* NeurIPS.
* I. Gibbs & E. Candès (2021). *Adaptive Conformal Inference Under Distribution Shift.* NeurIPS.
* MAPIE documentation: https://mapie.readthedocs.io

🐝 *Buzz says: you now have trust issues, in a good way. Next, in Lab 1.3: **why** did the model say that, and can a foundation model beat our trees?*